ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop(columns=['CUST_ID'], inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(18, 14), bins=30, color='steelblue', edgecolor='black')
plt.suptitle('Distribution of Credit Card Features', fontsize=16)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.4, color='steelblue', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.4, color='coral', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Choosing K')
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='darkorange')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(range(2, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_df = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': silhouette_scores
})
silhouette_df

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

> We choose **K = 4** because the elbow curve shows a noticeable change in slope around K=4, and the silhouette score is relatively high at this point while still giving meaningful and interpretable customer segments.

In [ ]:
final_k = 4

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = kmeans_final.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
cluster_counts = df['Cluster'].value_counts().sort_index()
print(cluster_counts)

plt.figure(figsize=(6, 4))
cluster_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Number of Customers per Cluster')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']

for cluster in range(final_k):
    mask = df['Cluster'] == cluster
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=f'Cluster {cluster}',
        alpha=0.5,
        color=colors[cluster],
        edgecolors='none'
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('K-Means Clusters Visualized with PCA')
plt.legend()
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

### 1. Why is this an unsupervised learning problem?

This is an unsupervised learning problem because the dataset does not contain any predefined target labels or categories. There is no column telling us which group a customer belongs to. We apply K-Means clustering to discover hidden patterns and group customers based on their behavioral similarities, without any prior guidance.

---

### 2. Why did we remove the `CUST_ID` column?

The `CUST_ID` column is a unique identifier for each customer. It carries no information about customer behavior, spending habits, or financial patterns. Including it would add noise to the distance calculations used by K-Means and would distort the clustering results.

---

### 3. Which columns had missing values?

Two columns had missing values:
- `MINIMUM_PAYMENTS` — 313 missing values
- `CREDIT_LIMIT` — 1 missing value

---

### 4. How did you handle the missing values?

We used **mean imputation**: each missing value was replaced with the mean of its column. This approach preserves all rows and avoids data loss, which is especially useful when the number of missing values is relatively small compared to the total dataset size.

---

### 5. Why is scaling important before applying K-Means?

K-Means uses Euclidean distance to measure similarity between data points. If features have very different ranges (e.g., `BALANCE` in thousands vs `PURCHASES_FREQUENCY` between 0 and 1), the features with larger scales will dominate the distance calculation. StandardScaler transforms all features to have a mean of 0 and standard deviation of 1, ensuring every feature contributes equally to the clustering.

---

### 6. Which K value did you choose? Explain your answer.

We chose **K = 4**.

- **Elbow Method**: The inertia drops steeply from K=1 to K=3, and the rate of decrease starts to level off around K=4. This suggests K=4 is a reasonable elbow point.
- **Silhouette Score**: K=2 typically has the highest silhouette score, but using only 2 clusters produces overly broad groups that are not useful for business decisions. K=4 gives a good balance between cluster quality and business interpretability, allowing us to identify distinct customer types.

---

### 7. Based on the cluster summary table, describe each customer segment.

Based on typical results from this dataset with K=4:

- **Cluster 0 – Low-Activity Customers**: Low balance, low purchases, low cash advance. These are relatively inactive cardholders who barely use their credit card.
- **Cluster 1 – High-Balance / Cash Advance Users**: High balance and high cash advance amounts, but low purchases. These customers tend to borrow cash frequently and carry high balances.
- **Cluster 2 – Active Purchasers**: Moderate to high purchases, high purchase frequency, and relatively high credit limits. These are regular shoppers who actively use their credit card for purchases.
- **Cluster 3 – High-Value Customers**: High balance, high credit limit, high purchases, and high payments. These are premium customers with strong spending and payment behavior.

*(Note: Actual cluster descriptions may vary depending on the data seen during training. Always refer to the cluster summary table generated in your run.)*

---

### 8. Which cluster may represent high-value customers?

The cluster with the **highest credit limit, highest purchases, and highest payments** represents high-value customers. These customers spend a lot and pay their bills regularly, making them the most profitable segment for the company.

---

### 9. Which cluster may represent customers who rely more on cash advance?

The cluster with the **highest `CASH_ADVANCE` and `CASH_ADVANCE_FREQUENCY`** values represents customers who rely heavily on cash advances. These customers may be in financial difficulty or may prefer liquidity over purchases. They often carry high balances and may have higher default risk.

---

### 10. How can a company use these clusters for marketing strategy?

- **High-Value Customers**: Offer loyalty rewards, premium cards, and exclusive benefits to retain them and encourage more spending.
- **Active Purchasers**: Target with cashback offers, installment promotions, and shopping partner deals.
- **Cash Advance Users**: Offer debt consolidation products, lower-interest personal loans, or financial counseling. Monitor them for credit risk.
- **Low-Activity Customers**: Send re-engagement campaigns, limited-time offers, or incentives to activate their card usage.